# Homework 2: Statistical Analysis of Jazz Solos

---

## Before you start

In the practice session we worked through a full statistical analysis of the Weimar Jazz Database together, asking questions like *"Does Parker play more notes than Davis?"* and *"Has jazz become more pitch-varied over time?"*

For this homework you will run a **parallel set of analyses** on the **same dataset**, but with **different performers, different features, and different questions**. The goal is to check that you can apply the workflow yourself and interpret the results in a short written answer.

## Files you need

1. `Weimars_solos_with_features_and_metadata.csv` — the features table (same as in the practice). We'll download it from github. 
2. The WJazzD MIDI files — we'll download these from the Jazzomat project website inside the notebook itself.

## How to submit

1. Complete **all six tasks** in this notebook.
2. Each task has:
   - A **code cell** where you write the analysis (you can add more if you need).
   - A **yellow box** where you write your short interpretation (1–3 sentences per question).
3. Save the notebook as `HW2_YourName.ipynb` and upload it to Moodle.

## Grading (6 points total)

| Task | Points | What we look for |
|------|:------:|------------------|
| 1. Descriptive statistics | 1 | Correct summary + correct plot |
| 2. Two-sample t-test | 1 | Assumption checks, correct test, Cohen's d, short interpretation |
| 3. Correlation | 1 | Pearson + Spearman, scatter plot, clear verbal answer |
| 4. One-way ANOVA | 1 | Correct ANOVA + Tukey HSD + η², interpretation |
| 5. Choose two scores and print their sheet music | 1 | Back to music21 |
| 6. Qualitative vs quantitative | 1 | Analyze your results both quantitatively and qualitatively |

---


## Setup

Run these cells first. They install and import everything you need.


In [ ]:
%pip install pandas numpy matplotlib seaborn scipy statsmodels music21 --quiet

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import os, urllib.request, zipfile

BG = "#F7F7F7"
plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "axes.edgecolor": "#888888",
    "axes.labelcolor": "#222222",
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "grid.color": "#CCCCCC",
    "grid.linestyle": "--",
    "grid.alpha": 0.5,
})

# Load and merge
data_file = "https://raw.githubusercontent.com/aljanaki/Digital_musicology/refs/heads/main/Practice%20sessions/Practice%202/Weimars_solos_with_features_and_metadata.csv"
df = pd.read_csv(data_file)
print("Dataframe:", df.shape)
print("Styles in corpus:", sorted(df['style'].dropna().unique()))

Dataframe: (456, 256)
Styles in corpus: ['BEBOP', 'COOL', 'FREE', 'FUSION', 'HARDBOP', 'POSTBOP', 'SWING', 'TRADITIONAL']


---
## Task 1 — Portrait of a performer

> **How "busy" are the solos of the most prolific performer in the corpus?**

Pick the performer with the **most solos** in `df` (you can find this with `df['performer'].value_counts()`). For that performer's solos only, produce:

1. A table of descriptive statistics (`.describe()`) for these four features: `event_density`, `pitch_range`, `abs_int_mean`, `avgtempo`.
2. A **histogram** of `event_density` for that performer's solos, with a vertical line at the performer's **median**.

Then briefly compare: is your performer **above or below** the corpus-wide median for each of those four features?


<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 1 — Your code goes below</b><br><br>
Fill in the cell(s) below.
</div>

In [ ]:
# Hint: start by finding the performer with the most solos
# top_performer = df['performer'].value_counts().idxmax()
# subset = df[df['performer'] == top_performer]
# subset[[...]].describe()

# Your code here


<div style="border: 2px solid #E1B64A; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #fff8e1;">
<b>📝 Your interpretation:</b><br><br>
<em>Which performer did you choose? How do their solos compare to the corpus average on these four features? Write 2–3 sentences here.</em>
</div>

---
## Task 2 — Coltrane vs. Rollins on interval size

> **Does John Coltrane play smaller (more stepwise) intervals on average than Sonny Rollins?**

Coltrane's late style is famous for its **scalar** "sheets of sound" — long runs of nearby notes. Rollins is famous for thematic motivic architecture. Run a t-test to compare `abs_int_mean` (mean absolute interval size) between these two performers. Beware of the small sample size.

Run a two-sample t-test following the pipeline from the practice:

1. Extract `abs_int_mean` for each performer.
2. Check **normality** (Shapiro–Wilk) and **equal variance** (Levene).
3. Run **Student's t** or **Welch's t** (depending on Levene).
4. Report **p-value** and **Cohen's d**.
5. Show a **violin plot** of the two groups.

- **H₀:** mean `abs_int_mean` is the same for the two performers.
- **H₁:** the means differ.


<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 2 — Your code goes below</b><br><br>
Fill in the cell(s) below.
</div>

In [ ]:
# Your code here


<div style="border: 2px solid #E1B64A; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #fff8e1;">
<b>📝 Your interpretation:</b><br><br>
<em>Is the difference statistically significant? In which direction (who has smaller intervals)? Is the effect size (Cohen's d) small, medium, or large? Does this match the musicological expectation? Write 2–4 sentences.</em>
</div>

---
## Task 3 — Tempo and note density

> **When a solo is played at a faster tempo, do performers play more notes per beat, or do they play fewer notes per beat?**

This is a genuine open question. You could argue it both ways:

- *More notes at faster tempo:* more beats per minute also means performers would use shorter rhythmic values and cram more notes into a beat.
- *Fewer notes at faster tempo:* players might use longer rhythmic values (half notes, quarters) at burnout tempos because sixteenth notes become physically impossible.

Let's compute a new feature - notes per beat.

In [ ]:
df["notes_per_beat"] = df["event_density"] * 60 / df["avgtempo"]

Compute **Pearson r** and **Spearman ρ** between `notes_per_beat` and `event_density`, make a scatter plot with a linear fit, and interpret.

<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 3 — Your code goes below</b><br><br>
Fill in the cell(s) below.
</div>

In [ ]:
# Your code here


<div style="border: 2px solid #E1B64A; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #fff8e1;">
<b>📝 Your interpretation:</b><br><br>
<em>What can you see on a scatterplot? Is it a linear relationship? What is the direction of the relationship? How strong is it (small / moderate / strong)? Do Pearson and Spearman agree? What does that tell you about the shape of the relationship? Write 2–3 sentences.</em>
</div>

---
## Task 4 — Event density across tempo classes

> **Does mean event density (notes per second) differ across the tempo classes SLOW / MEDIUM / UP?**

The metadata column `tempoclass` labels each solo as `SLOW`, `MEDIUM SLOW`, `MEDIUM`, `MEDIUM UP` or `UP`. First look at the distribution of this column and decide which categories have **enough solos** for a fair ANOVA (e.g. at least 15–20 each). **Group** the categories together (combine `SLOW` + `MEDIUM SLOW`, `MEDIUM` + `MEDIUM UP`, `UP`).

1. Create a tidy subset with 3 tempo groups that each have enough data.
2. Check Levene.
3. Run **one-way ANOVA**, compute **η²**.
4. Follow up with **Tukey HSD** if ANOVA is significant.
5. Make a boxplot of `event_density` by tempo group.

- **H₀:** mean `event_density` is equal across tempo groups.
- **H₁:** at least one tempo group's mean differs.


<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 4 — Your code goes below</b><br><br>
Fill in the cell(s) below.
</div>

In [ ]:
# Hint: start by inspecting df['tempoclass'].value_counts()

# Your code here


<div style="border: 2px solid #E1B64A; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #fff8e1;">
<b>📝 Your interpretation:</b><br><br>
<em>Which tempo group has the highest event density? Is that the result you expected? Which pairs of tempo groups differ significantly in the Tukey test? Write 3–4 sentences.</em>
</div>

---
## Task 5 - Look at the actual solos.
> **Pick two contrasting solos and display them.**

Download the WJazzD MIDI files (run the cell below — it pulls the zip from the Jazzomat project and extracts it). Then:

1. Pick **two solos** that contrast strongly — for example:
   - One **TRADITIONAL** or **SWING** solo (e.g. Louis Armstrong, Coleman Hawkins).
   - One **POSTBOP** solo (e.g. John Coltrane, Michael Brecker).
2. Load **both** MIDI files with `music21`.
3. For each, produce:
   - The **sheet music** display.

In [ ]:
# Run this once to download and unzip the MIDIs
MIDI_URL = "https://jazzomat.hfm-weimar.de/download/downloads/RELEASE2.0_mid_unquant.zip"
ZIP_PATH = "wjazz_midis.zip"
MIDI_DIR = "wjazz_midis"

if not os.path.exists(MIDI_DIR):
    print("Downloading MIDI archive...")
    urllib.request.urlretrieve(MIDI_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(MIDI_DIR)
    print("Done.")
else:
    print("MIDI folder already exists — skipping download.")

midi_files = []
for root, _, files in os.walk(MIDI_DIR):
    for f in files:
        if f.lower().endswith(".mid"):
            midi_files.append(os.path.join(root, f))
print(f"{len(midi_files)} MIDI files available.")

<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 5 — Your code goes below</b><br><br>
Fill in the cell(s) below. Feel free to add extra cells.
</div>

In [ ]:
# Hint: to find a solo by performer, filter the filenames

# Your code here


---
## Task 6 - Qualitative vs quantitative analysis
1. Pick one features of your choice. You can choose from event density, rhythm, interval range, or something else that was not discussed before from the features as described in [MeloSpy features](https://jazzomat.hfm-weimar.de/commandline_tools/melfeature/melfeature_features.html). 
2. Compare the features precomputed for you from the dataset.
3. Now, look at the actual scores and try to interpret what you see in the score qualitatively.
5. Deliberate on the differences between qualitative and quantitative analysis.

<div style="border: 2px solid #E1B64A; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #fff8e1;">
<b>📝 Your interpretation:</b><br><br>
<em>What feature did you choose? What are the values? Do the numerical feature of your choice <code>df</code> reflect what you see in the score? Write 4–6 sentences.</em>
</div>

---

## Wrap-up checklist

Before you submit, make sure:

- [ ] All six tasks have code that runs without errors (use **Runtime → Run all** to check).
- [ ] You wrote an interpretation in every yellow box.
- [ ] Your plots are labelled (axes, title).
- [ ] Your file is saved as `HW2_YourName.ipynb`.